# COM774 Coursework 1: cleaning the CI/CD runs dataset

**Goal:** turn a messy table of 303,079 software pipeline runs into a clean table a model can learn from in CW2.

**How to use this notebook**
1. Upload `final_research_dataset_MASTER.csv` (or the zip) to Colab and wait for the upload to finish.
2. Run the cells **in order, top to bottom**. Each cell depends on the ones above it.
3. Under every cell you will see an **Expected** line. If your output differs, stop and check.

**Story in one line:** look at the mess (Cells 1-7) → fix the labels and duplicates (8-12) → decide which rows to keep (13-14) → fix outliers and types (15-16) → turn words into numbers and add clues (17-21) → save (22).

## Cell 0: Check the file is complete
**What it does:** prints the file size and stops if it is too small.

**Why:** earlier, a half-uploaded file gave 1,962 rows instead of 303,079. Every number downstream would have been wrong. This check catches that immediately.

**Expected:** about 195 MB (Colab's `ls -lh` shows 186M).

In [ ]:
import os
path = "final_research_dataset_MASTER.csv"
size_mb = os.path.getsize(path) / 1e6
print(f"File size: {size_mb:.0f} MB")
assert size_mb > 180, "File looks incomplete: wait for the upload to finish, or upload it again."

## Cell 1: Load the data
**What it does:** reads the CSV into a table called `df`, and keeps an untouched copy called `df_raw`.

**Why:** pandas is the standard Python tool for tables. Keeping `df_raw` means we can always show "before and after" and never lose the original. `low_memory=False` stops pandas guessing column types in chunks.

**Say in the video:** "I kept the raw data untouched so every step can be compared to the original."

**Expected:** `Shape: (303079, 38)` and about 548 MB.

In [ ]:
import pandas as pd
import numpy as np

df_raw = pd.read_csv("final_research_dataset_MASTER.csv", low_memory=False)
df = df_raw.copy()   # keep the raw version untouched for before/after comparisons

print("Shape:", df.shape)
print(f"Memory: {df.memory_usage(deep=True).sum() / 1e6:.0f} MB")
df.head()

## Cell 2: Data types and missing values
**What it does:** `df.info()` lists every column, its type and how many values are filled. The second part turns that into "% missing", sorted worst first.

**Why:** you cannot justify a cleaning step without first showing the problem. This is the evidence for the dataset-overview slide.

**What to notice:** the percentages come in blocks (37.5%, 30%, 70%...). Real random gaps do not look like that. This is our first clue the file is several sources stuck together.

**Expected:** 303,079 rows, 38 columns, and `repo_name` about 92.5% missing.

In [ ]:
df.info()

missing = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
missing[missing > 0]

## Cell 3: Chart of the missing values
**What it does:** draws the missing percentages as a bar chart.

**Why:** a chart makes the problem obvious in two seconds. This is a good slide visual (slide 4).

In [ ]:
import matplotlib.pyplot as plt

missing[missing > 0].plot(kind="barh", figsize=(8, 7), color="steelblue")
plt.xlabel("% missing")
plt.title("Missing values by column (raw data)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Cell 4: The result column and duplicates
**What it does:** counts each value in `conclusion` (the outcome of the run), then counts repeated `run_id`s and fully identical rows.

**Why:** `conclusion` is what we want to predict, so it must be right. The counts reveal two problems: the same outcome written several ways (`success`/`passed`, `failure`/`failed`, `cancelled`/`canceled`) and 91,047 blanks. Also 50,529 run IDs repeat, but no row is fully identical, so the copies differ somewhere.

**Expected:** success 137,168 · NaN 91,047 · skipped 38,041 · passed 17,522 · failure 8,671 · cancelled 5,260 · failed 3,702 · errored 1,253 · startup_failure 271 · canceled 109 · action_required 35. Repeated run_ids 50,529. Fully duplicated rows 0.

In [ ]:
print(df["conclusion"].value_counts(dropna=False))
print("\nRepeated run_ids:", df["run_id"].duplicated().sum())
print("Fully duplicated rows:", df.duplicated().sum())

## Cell 5: Look at the duration column
**What it does:** shows summary statistics for `duration` (seconds) and counts runs longer than 24 hours.

**Why:** a typical run is about a minute, but the average is days. That gap means a few giant values are distorting everything.

**Expected:** median 60, mean about 203,147, max about 36.5 million seconds; 2,599 runs longer than 24 h.

In [ ]:
print(df["duration"].describe())
print("\nRuns longer than 24h:", (df["duration"] > 86400).sum())

## Cell 6: Is the file really three sources stuck together?
**What it does:** sorts every row into group A, B or C depending on which columns are filled in.

**Why:** the blocky missing values suggested three smaller tables were stacked. This cell tests that. If it is true, the blanks are **structural** (a source never recorded that column), which changes how we treat them.

**Say in the video:** "I diagnosed why data was missing before deciding what to do."

**Expected:** the three printed lines show A 189,446 · B 91,030 · C 22,586. The table below them shows the full groups: A 189,463 · B 91,030 · C 22,586, with nothing unassigned.

In [ ]:
has_commit = df["author_name"].notna()       # block A signal
has_outcome = df["conclusion"].notna()
has_old_commit = df["commit_date"].notna()   # block B signal
has_repo_name = df["repo_name"].notna()      # block C signal

print("A: commit + outcome       :", (has_commit & has_outcome).sum())
print("B: old-style commit only  :", (has_old_commit & ~has_outcome).sum())
print("C: outcome, no commit info:", (has_repo_name & ~has_commit).sum())

block = np.select(
    [has_commit, has_old_commit, has_repo_name],
    ["A", "B", "C"], default="unassigned")
print(pd.Series(block).value_counts())

## Cell 7: Are the huge durations errors?
**What it does:** (1) shows percentiles, so we see where "extreme" begins; (2) recomputes duration from the start and end times for the long runs; (3) draws raw vs log-scale histograms.

**Why:** we must know *why* values are huge before choosing a fix. They are not typing errors: `duration` equals `updated_at - created_at`, so it includes waiting and re-runs (one run was re-run 14 months later).

**Expected:** 99th percentile is far above the median; the `duration` and `calc` columns match (blank `calc` rows have no dates).

In [ ]:
import matplotlib.pyplot as plt

dur = df["duration"].dropna()
print(dur.quantile([0.5, 0.9, 0.95, 0.99, 0.999]))

t = df[df["duration"] > 86400].copy()
t["calc"] = (pd.to_datetime(t["updated_at"], errors="coerce")
             - pd.to_datetime(t["created_at"], errors="coerce")).dt.total_seconds()
print(t[["duration", "calc"]].head(10))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(dur.clip(upper=dur.quantile(0.99)), bins=60, color="indianred")
ax[0].set_title("Duration (raw, clipped at 99th pct)")
ax[1].hist(np.log1p(dur), bins=60, color="seagreen")
ax[1].set_title("Duration after log1p")
for a in ax: a.set_xlabel("seconds / log(seconds)")
plt.tight_layout(); plt.show()

## Cell 8: First attempt at tidying the labels
**What it does:** builds a lookup table (`label_map`) that translates every spelling to one standard word, and writes the result to a **new** column `conclusion_clean`. The original is left alone.

**Why:** a model would treat `passed` and `success` as different outcomes, although they mean the same. We also decided `errored` and `startup_failure` count as failures (a judgement call, say so).

**Note:** this only looks at `conclusion`. Cell 10 improves on it by also using `status`. The `label_map` is reused there.

**Expected:** success 154,690 · NaN 91,047 · skipped 38,041 · failure 13,897 · cancelled 5,369 · other 35.

In [ ]:
label_map = {
    "success": "success", "passed": "success",
    "failure": "failure", "failed": "failure",
    "errored": "failure", "startup_failure": "failure",
    "cancelled": "cancelled", "canceled": "cancelled",
    "skipped": "skipped", "action_required": "other",
}
df["conclusion_clean"] = df["conclusion"].str.strip().str.lower().map(label_map)
print(df["conclusion_clean"].value_counts(dropna=False))

## Cell 9: Why do some runs appear twice?
**What it does:** counts how many run IDs repeat and how many copies each has, then takes one example and shows only the columns where the two copies disagree.

**Why:** deleting duplicates blindly could throw away information. We first check what the copies look like. Result: every repeated run has exactly **two** copies, one full row and one thin "stub".

**Expected:** `50529 run_ids appear more than once`, and all have 2 copies.

In [ ]:
dups = df[df["run_id"].duplicated(keep=False)].sort_values("run_id")
print(dups["run_id"].nunique(), "run_ids appear more than once")
print(dups.groupby("run_id").size().value_counts())   # how many copies each

ex = dups["run_id"].iloc[0]
cmp = dups[dups["run_id"] == ex].T
cmp[cmp.nunique(axis=1, dropna=False) > 1]

## Cell 10: Find the real outcome for every run
**What it does:** uses `conclusion`, and where that is blank uses `status` instead, then applies the same `label_map` to make a clean `outcome` column.

**Why:** the three sources stored the result in different columns. Source B kept it in `status`. Using `conclusion` alone (Cell 8) would have thrown away about 40,000 usable runs.

**Expected:** success 223,827 · skipped 53,364 · failure 17,974 · cancelled 7,834 · NaN 44 · other 36.

In [ ]:
raw_outcome = df["conclusion"].fillna(df["status"])
df["outcome"] = raw_outcome.str.strip().str.lower().map(label_map)
print(df["outcome"].value_counts(dropna=False))

## Cell 11: Remove the duplicate stub rows
**What it does:** finds the thin stub rows (they have a commit date but no author) whose run ID also has a full row, and deletes only those.

**Why:** each duplicated run is the same run recorded twice. We keep the full copy, so nothing useful is lost, and we avoid counting the same run twice (which would bias a model).

**Expected:** `Removed 50,529 duplicate stub rows` and `Repeated run_ids left: 0`.

In [ ]:
has_full = df["author_name"].notna()          # block A marker
full_ids = set(df.loc[has_full, "run_id"])
is_stub = df["commit_date"].notna() & ~has_full   # block B marker

before = len(df)
df = df[~(is_stub & df["run_id"].isin(full_ids))].copy()
print(f"Removed {before - len(df):,} duplicate stub rows")
print("Repeated run_ids left:", df["run_id"].duplicated().sum())

## Cell 12: Keep only runs with a real pass or fail
**What it does:** keeps rows whose outcome is `success` or `failure`.

**Why:** skipped and cancelled runs never passed or failed, so they cannot teach a model "will it fail?". Filling in a guess would invent the very answer we want to predict.

**Say in the video:** "Skipped and cancelled runs have no real outcome, so I removed them rather than invent one."

**Expected:** `Rows: 252,550 -> 197,326`, success 182,331, failure 14,995, failure rate 7.6%.

In [ ]:
before = len(df)
df = df[df["outcome"].isin(["success", "failure"])].copy()
print(f"Rows: {before:,} -> {len(df):,}")
print(df["outcome"].value_counts())
print(f"Failure rate: {(df['outcome'] == 'failure').mean():.1%}")

## Cell 13: Which source filled in which columns?
**What it does:** labels each row A, B or C and shows, for each source, the % missing per column.

**Why:** this is the key evidence. Within each source, every column is either 0% or 100% missing, so the gaps are **structural, not random**. Filling them with averages would invent data. It also shows source C fails 22% of the time versus 6% for A.

**Expected:** A_full 146,110 · B_commit_only 28,739 · C_run_only 22,477. Fail rates A 6.1%, B 3.8%, C 22.0%.

In [ ]:
df["source"] = np.select(
    [df["author_name"].notna(), df["commit_date"].notna(), df["repo_name"].notna()],
    ["A_full", "B_commit_only", "C_run_only"], default="unknown")

print(df["source"].value_counts())
print(df.groupby("source")["outcome"].apply(lambda s: (s == "failure").mean()).round(3))

miss = df.drop(columns="source").isna().groupby(df["source"]).mean().mul(100).round(0).T
miss[(miss > 0).any(axis=1)]

## Cell 14: Keep source A and drop columns we cannot use
**What it does:** keeps only source A rows, then drops (1) columns that are empty in source A and (2) columns that contain the answer.

**Why:** source A is the only one with full details (commit size, files changed, run type...), which is what a model needs. Dropping `conclusion`, `status` and `conclusion_clean` stops the model cheating by reading the answer. The trade-off: only 32 repositories remain, which we state as a limitation.

**Expected:** `Rows: 197,326 -> 146,110`, `Columns: 41 -> 34`, success 137,168, failure 8,942.

In [ ]:
before_rows = len(df)
df = df[df["source"] == "A_full"].copy()

before_cols = df.shape[1]
df = df.drop(columns=[
    "repo_name", "commit_date", "author", "timestamp",   # empty in sheet A
    "conclusion", "status", "conclusion_clean",          # contain the answer
], errors="ignore")

print(f"Rows: {before_rows:,} -> {len(df):,}")
print(f"Columns: {before_cols} -> {df.shape[1]}")
print(df["outcome"].value_counts())

## Cell 15: Tame the extreme durations
**What it does:** (1) caps `duration` at its 99th percentile; (2) applies a log transform; (3) prints the mean before and after and draws two histograms.

**Why:** a few runs "lasted" months because the timer includes waiting and re-runs. Capping keeps every row but limits the damage. The log squeezes the scale so small and large values are comparable. We compared dropping, capping and logging and chose cap then log.

**Say in the video:** "I capped at the 99th percentile and applied a log transform so extreme values do not distort the model."

**Expected:** cap 70,390 s (19.6 hours); mean before 223,270, after cap 1,993.

In [ ]:
import matplotlib.pyplot as plt

cap = df["duration"].quantile(0.99)
df["duration_capped"] = df["duration"].clip(upper=cap)
df["duration_log"] = np.log1p(df["duration_capped"])

print(f"Cap value: {cap:,.0f} seconds ({cap/3600:.1f} hours)")
print("Mean before:", round(df["duration"].mean()))
print("Mean after cap:", round(df["duration_capped"].mean()))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(df["duration_capped"], bins=60, color="indianred")
ax[0].set_title("After cap (still very skewed)")
ax[1].hist(df["duration_log"], bins=60, color="seagreen")
ax[1].set_title("After cap + log (readable shape)")
for a in ax: a.set_xlabel("seconds / log(seconds)")
plt.tight_layout(); plt.show()

## Cell 16: Fix the data types
**What it does:** turns the five text dates into real dates, `True/False` into `1/0`, and whole numbers stored as decimals into integers.

**Why:** a computer cannot do maths on the text `"2025-02-24 16:42:12"`. Real dates let us calculate hours and ages later. Correct types also avoid silent errors.

**Expected:** 0 missing in each date column, date columns show `datetime64[ns, UTC]`.

In [ ]:
date_cols = ["created_at", "run_started_at", "updated_at",
             "authored_date", "committed_date"]
for c in date_cols:
    df[c] = pd.to_datetime(df[c], errors="coerce", utc=True)

df["is_merge"] = df["is_merge"].astype(str).str.lower().map({"true": 1, "false": 0}).astype(int)

whole_cols = ["run_number", "num_parents", "total_churn", "workflow_id",
              "run_attempt", "time_since_last_commit"]
df[whole_cols] = df[whole_cols].astype("int64")

print(df[date_cols].isna().sum())
print(df.dtypes.value_counts())
df[date_cols].dtypes

## Cell 17: Turn `event` into numbers (one-hot encoding)
**What it does:** groups rare event types (under 1,000 runs) into `other`, then makes one yes/no column per event type.

**Why:** models only understand numbers. Numbering the words 1, 2, 3 would wrongly suggest one is "bigger" than another, so we use separate yes/no columns. Grouping rare types avoids many nearly-empty columns.

**Expected:** 9 named types plus `other` (1,432); `Columns: 36 -> 45`.

In [ ]:
counts = df["event"].value_counts()
rare = counts[counts < 1000].index
df["event"] = df["event"].replace(rare, "other")

print(df["event"].value_counts())

before_cols = df.shape[1]
df = pd.get_dummies(df, columns=["event"], prefix="event", dtype=int)
print(f"\nColumns: {before_cols} -> {df.shape[1]}")
df.filter(like="event_").head()

## Cell 18: Remove ID and name columns
**What it does:** drops 14 columns that only label things (commit codes, author names and emails, free-text message, `source`) or repeat other columns (`total_churn` = additions + deletions).

**Why:** names and codes identify *who* or *which one*, not *how risky*. A model would memorise them, and encoding them would create thousands of useless columns.

**Expected:** `Columns: 45 -> 31`. (If you run it twice, the second run prints 31 -> 31, which is harmless.)

In [ ]:
drop_cols = [
    "commit_sha", "head_sha", "node_id", "head_branch",
    "author_name", "author_email", "committer_name", "committer_email",
    "actor_login", "triggering_actor_login", "message",
    "source", "total_churn", "timestamp_raw",
]
before = df.shape[1]
df = df.drop(columns=drop_cols, errors="ignore")
print(f"Columns: {before} -> {df.shape[1]}")

## Cell 19: Encode `repo`
**What it does:** same method as Cell 17 for the repository name, grouping repositories with under 1,000 runs into `other_repo`.

**Why:** repository genuinely matters: failure rates range from 0.5% to 43% between repositories. So we keep it as a feature.

**Expected:** `Columns: 31 -> 55`, and 8 small repositories listed (actions/cache, pallets/flask, actions/setup-node, opencv/opencv, actions/upload-artifact, actions/checkout, kubernetes/kubernetes, golang/go).

In [ ]:
counts = df["repo"].value_counts()
small = counts[counts < 1000].index
df["repo"] = df["repo"].replace(small, "other_repo")

before = df.shape[1]
df = pd.get_dummies(df, columns=["repo"], prefix="repo", dtype=int)
print(f"Columns: {before} -> {df.shape[1]}")
print(counts[counts < 1000])   # which repos were grouped

## Cell 20: Feature engineering (new clues from the dates)
**What it does:** creates the hour of day, weekday, a weekend flag, and the age of the code change when the run started (on a log scale), then drops the raw date columns and the old duration columns.

**Why:** a model cannot use a raw timestamp, but it can use "what hour" or "how old was the change". Negative ages (clock differences) are set to 0.

**Expected:** `Columns: 59 -> 52` (4 added first, then 7 dropped).

In [ ]:
df["created_hour"] = df["created_at"].dt.hour
df["created_weekday"] = df["created_at"].dt.dayofweek    # 0 = Monday ... 6 = Sunday
df["is_weekend"] = (df["created_weekday"] >= 5).astype(int)

age = (df["created_at"] - df["committed_date"]).dt.total_seconds().clip(lower=0)
df["commit_age_log"] = np.log1p(age)

before = df.shape[1]
df = df.drop(columns=["created_at", "run_started_at", "updated_at",
                      "authored_date", "committed_date",
                      "duration", "duration_capped"])
print(f"Columns: {before} -> {df.shape[1]}")
df[["created_hour", "created_weekday", "is_weekend", "commit_age_log"]].head()

## Cell 21: Scaling
**What it does:** (1) drops two more useless columns; (2) log-transforms five very skewed columns; (3) standardises all number columns to mean 0 and spread 1.

**Why:** `additions` reaches 1.4 million while `created_hour` stops at 23. Without scaling, a model can treat bigger numbers as more important. The log tames extreme skew (`additions` skew 70.5 to 0.7). Standard scaling puts every column on the same footing. Yes/no columns are left alone.

**For CW2:** fit the scaler on the training data only, then apply it to the test data, to avoid leakage.

**Expected:** first line `True`; skew `additions` 70.5 -> 0.7; `Columns: 50`; means about 0 and std about 1.

In [ ]:
from sklearn.preprocessing import StandardScaler

print("num_parents is just is_merge + 1:", (df["num_parents"] == df["is_merge"] + 1).all())
df = df.drop(columns=["workflow_id", "num_parents"])

skewed = ["additions", "deletions", "files_modified", "msg_len", "time_since_last_commit"]
print("Skew BEFORE:\n", df[skewed].skew().round(1))
df[skewed] = np.log1p(df[skewed])
print("\nSkew AFTER:\n", df[skewed].skew().round(1))

scale_cols = skewed + ["run_number", "run_attempt", "duration_log",
                       "commit_age_log", "created_hour", "created_weekday"]
scaler = StandardScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])

print(f"\nColumns: {df.shape[1]}")
df[scale_cols].agg(["mean", "std"]).round(2)

## Cell 22: Set the target, final checks, save
**What it does:** turns `outcome` into a 0/1 `target` (1 = failed), runs final quality checks and saves `cleaned_cicd_dataset.csv`.

**Why:** the model needs a numeric answer to predict. The checks prove the data is ready: no missing values, no repeated runs, no text columns. The saved file is your CW1 result and the starting point for CW2.

**Expected:** shape (146110, 50), 0 missing, 0 repeated run_ids, non-numeric columns `[]`, target 0: 137,168 and 1: 8,942, failure rate 6.1%.

In [ ]:
df["target"] = (df["outcome"] == "failure").astype(int)
df = df.drop(columns=["outcome"])

print("Rows, columns:", df.shape)
print("Missing values:", int(df.isna().sum().sum()))
print("Repeated run_ids:", int(df["run_id"].duplicated().sum()))
print("Non-numeric columns:", df.select_dtypes(exclude="number").columns.tolist())
print("\nTarget balance:")
print(df["target"].value_counts())
print(f"Failure rate: {df['target'].mean():.1%}")

df.to_csv("cleaned_cicd_dataset.csv", index=False)
try:
    from google.colab import files
    files.download("cleaned_cicd_dataset.csv")
except ImportError:
    print("Saved cleaned_cicd_dataset.csv")